# Agentic AI Systems — Worklet 13
# Research Grant Management System (apniLeap AURA)
## Lab 3 · Skill Creation — Reusable Plan & Format Skills

**Phase A · Foundations** &nbsp;|&nbsp; **Goal:** Package repeatable capability as reusable, schema-typed skills

---

**Description**

A skill is not a prompt string; it is a reusable, schema-constrained software unit.
This lab implements two core skills of the Research Grant Management System:
1. `plan_skill`: Transforms intake parameters and call requirements into a validated,
   locked execution plan with phased agent assignments and acceptance criteria.
2. `format_skill`: Renders the proposal narrative, itemized budget CSV, and compliance
   checklist into standard agency-submission documents.

**Key concepts demonstrated**

- Explicit typed schemas (input & output specifications).
- Plan locking and SHA-256 integrity checksum.
- Cross-domain reusability test (verifying that the exact same skill handles
  Government, Industry/CSR, and International grant calls without code edits).
- Strict contract enforcement: rejection of plans missing mandatory units (objectives, references).

**Assignments included**

- Assignment 1: Multi-domain template synthesis (handling Government vs Industry CSR).
- Assignment 2: Low-input / contract validation test (ensuring invalid plans raise structured errors).

---

*Runs standalone on Python 3.10+ (Jupyter or Google Colab): no API key, no network, standard library only. Source script: [`scripts/lab03_skill_creation.py`](./scripts/lab03_skill_creation.py).*

### Cell 1 · Shared institutional seed data

`seed_data.py` is written to disk so the notebook runs standalone. It is the Python mirror of the course project's `backend/src/seed.js`: funding calls, literature corpus, policies, personnel rate card, users and the sanctioned-project portfolio. *(Collapse — shared by all labs.)*

In [1]:
%%writefile seed_data.py
"""
AURA Seed Data & Corpus -- Research Grant Management System
Corresponds to backend/src/seed.js, faculty.js, and projects.js
"""

INSTITUTION = {
    "id": "inst-mist",
    "name": "Meridian Institute of Science and Technology",
    "short": "MIST",
    "type": "university",
    "city": "Pune",
    "country": "IN",
    "researchOffice": "Research & Development Office",
    "signatory": "Dean, Research & Development",
}

USERS = [
    {
        "id": "usr-krishnan",
        "name": "Dr. Uma Mudengudi",
        "email": "uma.mudengudi@mist.edu.in",
        "role": "dean",
        "title": "Dean, Research & Development",
        "department": "Office of Dean R&D",
        "permissions": ["run:create", "run:read:all", "review:approve", "package:view", "audit:read", "version:rollback", "metrics:read"],
    },
    {
        "id": "usr-suneeta-v-budihal",
        "name": "Dr. Suneeta V Budihal",
        "email": "suneeta.v.budihal@mist.edu.in",
        "role": "pi",
        "title": "Associate Professor",
        "department": "SoECE",
        "permissions": ["run:create", "run:read:own", "draft:edit", "package:view"],
    },
    {
        "id": "usr-satyadhyan-chickerur",
        "name": "Dr. Satyadhyan Chickerur",
        "email": "satyadhyan.chickerur@mist.edu.in",
        "role": "pi",
        "title": "Professor & Head",
        "department": "CSE",
        "permissions": ["run:create", "run:read:own", "draft:edit", "package:view"],
    },
    {
        "id": "usr-research-office",
        "name": "Research & Development Office",
        "email": "grants.cell@mist.edu.in",
        "role": "research_office",
        "title": "Grants Cell & Compliance",
        "department": "Office of Dean R&D",
        "permissions": ["run:create", "run:read:all", "review:approve", "package:view", "audit:read", "version:rollback"],
    },
]

RATE_CARD = {
    "institutionId": "inst-mist",
    "overheadPct": 15,
    "contingencyPct": 5,
    "salaryEscalationPct": 5,
    "equipmentMaintenancePct": 4,
    "effectiveFrom": "2026-04-01",
    "note": "Institutional default rates. Agency-published rates for a call always take precedence.",
}

PERSONNEL = [
    {"id": "per-uma", "name": "Dr. Uma Mudengudi", "role": "PI", "title": "Dean (R&D) & Professor", "department": "SoECE", "monthlyCost": 0, "defaultEffortPct": 10, "careerStage": "professor"},
    {"id": "per-suneeta", "name": "Dr. Suneeta V Budihal", "role": "PI", "title": "Faculty Member", "department": "SoECE", "monthlyCost": 0, "defaultEffortPct": 10, "careerStage": "associate"},
    {"id": "per-satyadhyan", "name": "Dr. Satyadhyan Chickerur", "role": "PI", "title": "Faculty Member", "department": "CSE", "monthlyCost": 0, "defaultEffortPct": 10, "careerStage": "associate"},
    {"id": "per-ra", "name": "Research Associate (to be appointed)", "role": "RA", "title": "Research Associate", "department": "Research Office", "monthlyCost": 48000, "defaultEffortPct": 100, "careerStage": "n/a"},
    {"id": "per-phd", "name": "Doctoral Scholar (to be appointed)", "role": "PhD", "title": "Doctoral Scholar", "department": "Research Office", "monthlyCost": 36000, "defaultEffortPct": 100, "careerStage": "n/a"},
]

POLICIES = [
    {
        "id": "pol-overhead",
        "title": "Institutional overhead and costing policy",
        "tags": ["budget", "overhead"],
        "text": "Overheads are charged at the rate published by the funding agency for the call. Where the agency publishes no rate, the institutional default of 15% of total direct costs applies. Overhead may not fund permanent faculty salaries.",
    },
    {
        "id": "pol-ethics",
        "title": "Human subjects research ethics policy",
        "tags": ["compliance", "ethics", "health"],
        "text": "Any proposal involving human participants, identifiable health data or clinical validation requires Institutional Ethics Committee approval before the award is activated. A conditional certificate may be attached at submission and must be converted to full approval before funds are released.",
    },
    {
        "id": "pol-data",
        "title": "Research data protection policy (DPDP-aligned)",
        "tags": ["compliance", "data", "privacy"],
        "text": "Proposals collecting personal data must include a Data Management Plan covering lawful basis, de-identification, storage location, retention period and a named data steward. Cross-border transfer of health data requires a documented risk assessment.",
    },
    {
        "id": "pol-ai",
        "title": "Disclosure of AI-assisted writing",
        "tags": ["compliance", "ai", "authorship"],
        "text": "AI-assisted drafting must be disclosed in the proposal cover declaration and in the institutional submission record. AI-generated text is a draft until a named faculty member reviews and accepts it; AI tools may not be listed as authors or investigators.",
    },
    {
        "id": "pol-coi",
        "title": "Conflict of interest declaration",
        "tags": ["compliance", "coi"],
        "text": "Every investigator must declare financial interests, advisory roles and institutional relationships that could reasonably be perceived to influence the research. Declarations are collected by the Research Office and retained for the audit period.",
    },
    {
        "id": "pol-signoff",
        "title": "Authorised signatory policy",
        "tags": ["compliance", "approval"],
        "text": "Only the Dean, Research & Development, or a delegated Research Office signatory, may authorise submission to a funding agency. The authorising officer must record the decision, the version reviewed, and any conditions attached to approval.",
    },
]

COMPLIANCE_RULES = [
    {"id": "rule-sections", "kind": "sections", "appliesTo": "all", "severity": "blocking", "requirement": "Every section required by the call is present and non-empty.", "evidence": "Call required-section list vs draft section index."},
    {"id": "rule-pages", "kind": "pageLimit", "appliesTo": "all", "severity": "blocking", "requirement": "The narrative stays within the page limit published for the call.", "evidence": "Word count converted at 500 words per page."},
    {"id": "rule-budget-ceiling", "kind": "budgetCeiling", "appliesTo": "all", "severity": "blocking", "requirement": "The requested total does not exceed the agency ceiling for this call.", "evidence": "Budget total vs call award maximum."},
    {"id": "rule-equipment-cap", "kind": "equipmentCap", "appliesTo": "all", "severity": "blocking", "requirement": "Equipment does not exceed the percentage ceiling in the call guidelines.", "evidence": "Equipment subtotal / total requested."},
    {"id": "rule-salary-cap", "kind": "salaryCap", "appliesTo": "all", "severity": "blocking", "requirement": "Manpower costs stay within the call salary ceiling.", "evidence": "Personnel subtotal / total requested."},
    {"id": "rule-overhead", "kind": "overheadRate", "appliesTo": "all", "severity": "blocking", "requirement": "Overhead is charged at exactly the rate published for the call.", "evidence": "Overhead amount vs rate x direct costs."},
    {"id": "rule-ethics", "kind": "document", "docKey": "ethics", "appliesTo": "all", "severity": "blocking", "requirement": "Ethics clearance is attached where the call or the research involves human participants.", "evidence": "Document register vs call requirement."},
    {"id": "rule-coi", "kind": "document", "docKey": "coi", "appliesTo": "all", "severity": "blocking", "requirement": "Conflict-of-interest declarations are attached for all investigators.", "evidence": "Document register vs call requirement."},
    {"id": "rule-dmp", "kind": "document", "docKey": "dmp", "appliesTo": "all", "severity": "blocking", "requirement": "A Data Management Plan is attached where required.", "evidence": "Document register vs call requirement."},
    {"id": "rule-disclosure", "kind": "aiDisclosure", "appliesTo": "all", "severity": "blocking", "requirement": "AI-assisted drafting is disclosed in the submission declaration.", "evidence": "Institutional AI disclosure policy vs package declaration."},
]

DOCUMENT_REGISTER = [
    {"key": "ethics", "label": "Institutional Ethics Committee clearance", "status": "conditional", "note": "Conditional certificate available; full approval pending protocol review."},
    {"key": "coi", "label": "Conflict-of-interest declarations (all investigators)", "status": "attached", "note": "Signed by all investigators."},
    {"key": "dmp", "label": "Data Management Plan", "status": "attached", "note": "Deposited in the institutional DMP registry."},
    {"key": "budget-justification", "label": "Budget justification", "status": "attached", "note": "Generated by the Budget Planning Agent."},
    {"key": "csr-1", "label": "CSR-1 registration and company CSR policy", "status": "pending", "note": "Company secretarial certificate requested."},
    {"key": "consortium-agreement", "label": "Consortium agreement and partner letters", "status": "pending", "note": "Letters of intent collected; agreement under legal review."},
]

CALLS = [
    {
        "id": "call-serb-crg",
        "agency": "SERB",
        "agencyFull": "Science and Engineering Research Board, DST",
        "program": "Core Research Grant (CRG)",
        "title": "SERB Core Research Grant 2026 -- Individual Investigator",
        "fundingType": "government",
        "domains": ["ai-ml", "energy", "materials", "health", "climate"],
        "tags": ["artificial intelligence", "machine learning", "federated learning", "sustainable energy", "materials science", "medical imaging"],
        "deadline": "2026-12-15",
        "budgetMin": 3000000,
        "budgetMax": 7500000,
        "durationMonths": 36,
        "pageLimit": 25,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute"], "consortiumRequired": False, "minPartners": 0, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 15, "salaryCapPct": 45, "equipmentCapPct": 40, "travelCapPct": 10, "contingencyPct": 5},
        "guidelines": [
            {"id": "call-serb-crg#g1", "section": "submission", "text": "Proposals must be routed through the institutional research office; individual submissions are not accepted."},
            {"id": "call-serb-crg#g2", "section": "budget", "text": "The total request may not exceed INR 75,00,000 for a duration of up to 36 months."},
            {"id": "call-serb-crg#g3", "section": "budget", "text": "Equipment may not exceed 40% of the total requested budget; requests above this ceiling require a written justification from the Head of Institution."},
            {"id": "call-serb-crg#g4", "section": "budget", "text": "Overheads are payable at 15% of total direct costs."},
            {"id": "call-serb-crg#g5", "section": "documents", "text": "A Data Management Plan and a conflict-of-interest declaration are mandatory at submission."},
            {"id": "call-serb-crg#g6", "section": "format", "text": "The narrative may not exceed 25 pages excluding references and the budget annexure."},
        ],
    },
    {
        "id": "call-icmr-adhoc",
        "agency": "ICMR",
        "agencyFull": "Indian Council of Medical Research",
        "program": "Ad-hoc Research Project",
        "title": "ICMR Ad-hoc Research Project 2026 -- Diagnostics & Public Health",
        "fundingType": "government",
        "domains": ["health", "ai-ml", "public-health"],
        "tags": ["diabetic retinopathy", "medical imaging", "rural health", "screening", "federated learning", "digital health", "primary care"],
        "deadline": "2026-12-05",
        "budgetMin": 2000000,
        "budgetMax": 6000000,
        "durationMonths": 36,
        "pageLimit": 25,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute", "hospital"], "consortiumRequired": False, "minPartners": 0, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 15, "salaryCapPct": 50, "equipmentCapPct": 30, "travelCapPct": 10, "contingencyPct": 5},
        "guidelines": [
            {"id": "call-icmr-adhoc#g1", "section": "eligibility", "text": "Applicants must hold a regular position in a recognised institution or hospital and demonstrate access to the study population."},
            {"id": "call-icmr-adhoc#g2", "section": "budget", "text": "The total request must fall between INR 20,00,000 and INR 60,00,000; equipment is capped at 30%."},
            {"id": "call-icmr-adhoc#g3", "section": "ethics", "text": "Institutional Ethics Committee approval is mandatory for any study involving human participants or their data."},
            {"id": "call-icmr-adhoc#g4", "section": "data", "text": "A Data Management Plan describing de-identification and retention is required; identifiable health data may not leave the institution without a documented risk assessment."},
            {"id": "call-icmr-adhoc#g5", "section": "deliverables", "text": "The proposal must state measurable clinical or public-health outcomes for the funding period."},
        ],
    },
    {
        "id": "call-tata-csr",
        "agency": "Tata Trusts",
        "agencyFull": "Tata Trusts -- Corporate Social Responsibility",
        "program": "CSR Research Partnership",
        "title": "Tata Trusts CSR Research Partnership 2026 -- Health & Education",
        "fundingType": "industry",
        "domains": ["health", "education", "public-health"],
        "tags": ["community health", "digital health", "education technology", "rural outreach", "screening"],
        "deadline": "2026-11-15",
        "budgetMin": 2500000,
        "budgetMax": 10000000,
        "durationMonths": 24,
        "pageLimit": 20,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute", "hospital", "ngo"], "consortiumRequired": False, "minPartners": 0, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 8, "salaryCapPct": 55, "equipmentCapPct": 25, "travelCapPct": 10, "contingencyPct": 3},
        "guidelines": [
            {"id": "call-tata-csr#g1", "section": "compliance", "text": "The activity must fall within Schedule VII of the Companies Act 2013 and the implementing partner must hold a valid CSR-1 registration."},
            {"id": "call-tata-csr#g2", "section": "budget", "text": "Grants range from INR 25,00,00,000 to INR 1,00,00,000 over 24 months; overheads are capped at 8%."},
            {"id": "call-tata-csr#g3", "section": "reporting", "text": "Quarterly outcome reporting against agreed indicators is a condition of release of the second tranche."},
        ],
    },
    {
        "id": "call-eu-horizon",
        "agency": "EU Horizon Europe",
        "agencyFull": "European Commission -- Horizon Europe",
        "program": "Cluster 4: Digital, Industry and Space",
        "title": "Horizon Europe 2027 -- Trustworthy AI for Health Systems (RIA)",
        "fundingType": "international",
        "domains": ["ai-ml", "health", "digital"],
        "tags": ["trustworthy ai", "federated learning", "health systems", "digital health", "data governance", "open science"],
        "deadline": "2027-01-20",
        "budgetMin": 18000000,
        "budgetMax": 45000000,
        "durationMonths": 36,
        "pageLimit": 45,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute", "hospital"], "consortiumRequired": True, "minPartners": 3, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 25, "salaryCapPct": 40, "equipmentCapPct": 30, "travelCapPct": 15, "contingencyPct": 0},
        "guidelines": [
            {"id": "call-eu-horizon#g1", "section": "eligibility", "text": "Proposals require a consortium of at least three independent legal entities from three different countries."},
            {"id": "call-eu-horizon#g2", "section": "budget", "text": "Personnel costs may not exceed 40% and equipment 30% of the total eligible cost; a 25% flat rate covers indirect costs."},
            {"id": "call-eu-horizon#g3", "section": "open-science", "text": "Open science practices, FAIR data management and a gender and inclusion plan are evaluated under the impact criterion."},
        ],
    },
    {
        "id": "call-serb-supra-expired",
        "agency": "SERB",
        "agencyFull": "Science and Engineering Research Board, DST",
        "program": "SUPRA",
        "title": "SERB SUPRA 2026 -- Closed",
        "fundingType": "government",
        "domains": ["ai-ml", "health"],
        "tags": ["artificial intelligence", "medical imaging"],
        "deadline": "2026-06-30",
        "budgetMin": 1000000,
        "budgetMax": 3000000,
        "durationMonths": 24,
        "status": "closed",
        "verified": True,
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university"], "consortiumRequired": False, "minPartners": 0},
        "budgetRules": {"overheadPct": 15, "salaryCapPct": 45, "equipmentCapPct": 40, "travelCapPct": 10, "contingencyPct": 5},
        "guidelines": [{"id": "call-serb-supra-expired#g1", "section": "status", "text": "This call closed on 30 June 2026 and is retained for reference only."}],
    },
    {
        "id": "call-unverified-sample",
        "agency": "Unknown Aggregator",
        "agencyFull": "Third-party opportunity aggregator",
        "program": "Sample Aggregator Call",
        "title": "Unverified Grant Opportunity 2027",
        "fundingType": "government",
        "domains": ["engineering"],
        "tags": ["engineering"],
        "deadline": "2027-02-01",
        "budgetMin": 800000,
        "budgetMax": 2500000,
        "durationMonths": 24,
        "verified": False,
        "status": "unverified",
        "eligibility": {"careerStages": ["assistant"], "institutionTypes": ["college"], "consortiumRequired": False, "minPartners": 0},
        "budgetRules": {"overheadPct": 10, "salaryCapPct": 40, "equipmentCapPct": 35, "travelCapPct": 12, "contingencyPct": 5},
        "guidelines": [{"id": "call-unverified#g1", "section": "status", "text": "Listing could not be confirmed against an official agency notification."}],
    },
]

LITERATURE = [
    {
        "id": "lit-001",
        "title": "Federated learning for diabetic retinopathy screening across rural clinics",
        "authors": ["S. Banerjee", "R. Iyer", "M. Kulkarni"],
        "year": 2024,
        "venue": "Journal of Medical AI Research",
        "tags": ["federated learning", "diabetic retinopathy", "rural health", "medical imaging", "screening"],
        "abstract": "A four-site federated study reports comparable sensitivity to centralised training while keeping 1.2 million fundus images inside participating clinics.",
        "citations": 61,
        "doi": "10.5555/aura.demo.001",
    },
    {
        "id": "lit-002",
        "title": "Deep learning detection of referable diabetic retinopathy in primary care",
        "authors": ["L. Fernandes", "A. Gupta"],
        "year": 2023,
        "venue": "Ophthalmology Informatics",
        "tags": ["diabetic retinopathy", "medical imaging", "primary care", "screening", "deep learning"],
        "abstract": "A prospective primary-care evaluation of 8,400 patients reports 92% sensitivity and 88% specificity for referable retinopathy.",
        "citations": 148,
        "doi": "10.5555/aura.demo.002",
    },
    {
        "id": "lit-003",
        "title": "Privacy-preserving federated learning in healthcare: a systematic review",
        "authors": ["H. Park", "J. Okafor", "N. Singh"],
        "year": 2024,
        "venue": "Health Data Science Review",
        "tags": ["federated learning", "privacy", "health systems", "data governance"],
        "abstract": "Reviews 74 studies and finds that differential privacy and secure aggregation are rarely combined in clinical deployments.",
        "citations": 96,
        "doi": "10.5555/aura.demo.003",
    },
    {
        "id": "lit-004",
        "title": "Non-mydriatic fundus photography in resource-limited settings",
        "authors": ["P. Adjei", "T. Nguyen"],
        "year": 2022,
        "venue": "Global Eye Health",
        "tags": ["rural health", "screening", "medical imaging", "primary care"],
        "abstract": "Image quality is the dominant failure mode for non-mydriatic cameras operated by trained community health workers.",
        "citations": 57,
        "doi": "10.5555/aura.demo.004",
    },
    {
        "id": "lit-005",
        "title": "Teleophthalmology screening pathways in low-resource districts",
        "authors": ["K. Rao", "D. Bhattacharya"],
        "year": 2023,
        "venue": "Indian Journal of Public Health",
        "tags": ["screening", "rural health", "health systems", "telemedicine"],
        "abstract": "Hub-and-spoke teleophthalmology reduced time-to-diagnosis from 26 to 9 days across six districts.",
        "citations": 73,
        "doi": "10.5555/aura.demo.005",
    },
]

HISTORY = [
    {
        "id": "hist-retina-2024",
        "title": "AI-assisted retinal screening for diabetic retinopathy in Pune district",
        "agency": "ICMR",
        "year": 2024,
        "outcome": "not-funded",
        "amount": 5600000,
        "tags": ["diabetic retinopathy", "screening", "rural health", "medical imaging"],
        "abstract": "Proposed a three-site pilot of automated retinal grading with referral support for 12,000 patients.",
        "lessons": "Reviewers questioned the missing ethics clearance and the absence of a site-level data-governance plan. Both are now prerequisites.",
    },
    {
        "id": "hist-fl-testbed-2023",
        "title": "Federated learning testbed for multi-hospital medical imaging",
        "agency": "SERB",
        "year": 2023,
        "outcome": "funded",
        "amount": 6200000,
        "tags": ["federated learning", "medical imaging", "privacy", "testbed"],
        "abstract": "Built a three-hospital federated learning testbed with secure aggregation and site-level audit logging.",
        "lessons": "Delivered the testbed; clinical validation was out of scope and remains the open gap.",
    },
]


Writing seed_data.py


### Cell 2 · Imports and fixtures

In [2]:
import hashlib
import json
from typing import Dict, Any, List, Optional
from seed_data import CALLS, INSTITUTION


# ==============================================================================
# Skill 1: Plan Skill (Specification, Lock, and Revision)
# ==============================================================================

PLAN_SCHEMA = {
    "name": "plan_skill",
    "version": "1.0.0",
    "description": "Expands research brief and matched call into a locked execution plan.",
    "required_input_fields": ["researchArea", "callId", "domain"],
    "required_output_fields": ["id", "units", "phases", "acceptance", "locked", "checksum"],
}

SECTION_SPECS = {
    "abstract": {"title": "Abstract", "min_words": 90, "purpose": "Problem, approach, outcomes"},
    "objectives": {"title": "Objectives & Aims", "min_words": 90, "purpose": "Measurable targets"},
    "background": {"title": "State of the Art & Gaps", "min_words": 130, "purpose": "Related work"},
    "methodology": {"title": "Methodology", "min_words": 160, "purpose": "Technical design & evaluation"},
    "timeline": {"title": "Work Plan & Milestones", "min_words": 100, "purpose": "Deliverable schedule"},
    "outcomes": {"title": "Expected Outcomes", "min_words": 110, "purpose": "Dissemination & IP"},
    "data-management": {"title": "Data Management & Ethics", "min_words": 110, "purpose": "DPDP compliance"},
    "budget-justification": {"title": "Budget Justification", "min_words": 80, "purpose": "Basis for costs"},
    "references": {"title": "References", "min_words": 30, "purpose": "Resolvable citations"},
    # Industry domain specific sections
    "executive-summary": {"title": "Executive Summary", "min_words": 90, "purpose": "CSR executive briefing"},
    "need-assessment": {"title": "Need Assessment", "min_words": 130, "purpose": "Beneficiary analysis"},
    "sustainability": {"title": "Sustainability Plan", "min_words": 100, "purpose": "Post-grant continuity"},
    "compliance": {"title": "Statutory Compliance", "min_words": 90, "purpose": "Schedule VII CSR declaration"},
}

DEFAULT_ACCEPTANCE_CHECKS = [
    "sections_complete",
    "sections_min_words",
    "claims_grounded",
    "citations_resolve",
    "budget_reconciles",
    "budget_caps",
    "compliance_blocking",
]

### Cell 3 · `apply_plan_skill` (function)

In [3]:
def apply_plan_skill(inputs: Dict[str, Any]) -> Dict[str, Any]:
    """
    Executes the Plan Skill: validates inputs, expands sections, binds agents,
    and locks the plan with an immutable cryptographic checksum.
    """
    for req in PLAN_SCHEMA["required_input_fields"]:
        if req not in inputs or not inputs[req]:
            raise ValueError(f"Plan skill error: missing required input '{req}'")

    research_area = inputs["researchArea"]
    call_id = inputs["callId"]
    domain = inputs["domain"]
    sections = inputs.get("sections", [
        "abstract", "objectives", "background", "methodology",
        "timeline", "outcomes", "data-management", "budget-justification", "references"
    ])

    units = []
    for i, sec_key in enumerate(sections, 1):
        meta = SECTION_SPECS.get(sec_key, {"title": sec_key.replace("-", " ").title(), "min_words": 80, "purpose": ""})
        units.append({
            "id": f"U{i:02d}",
            "key": sec_key,
            "title": meta["title"],
            "purpose": meta["purpose"],
            "min_words": meta["min_words"],
            "agent": "proposal_drafting",
            "required": True,
            "evidence_required": True,
        })

    phases = [
        {"id": "gate-1", "label": "Profile intake", "agent": "coordinator", "mode": "sequential"},
        {"id": "discovery", "label": "Discovery & Literature", "agent": "funding+literature", "mode": "parallel"},
        {"id": "drafting", "label": "Proposal narrative", "agent": "drafting", "mode": "sequential"},
        {"id": "budget-compliance", "label": "Budget & Compliance", "agent": "budget+compliance", "mode": "parallel"},
        {"id": "validation", "label": "Validation & Repair", "agent": "coordinator", "mode": "sequential"},
        {"id": "packaging", "label": "Submission packaging", "agent": "submission", "mode": "sequential"},
        {"id": "approval", "label": "Dean R&D human approval", "agent": "dean_human_gate", "mode": "sequential"},
    ]

    plan = {
        "id": f"PLAN-{domain.upper()}-{abs(hash(research_area)) % 10000:04d}",
        "version": 1,
        "title": f"Grant Proposal Plan -- {research_area}",
        "callId": call_id,
        "domain": domain,
        "units": units,
        "phases": phases,
        "acceptance": list(DEFAULT_ACCEPTANCE_CHECKS),
        "locked": False,
        "checksum": None,
    }

    return lock_plan(plan)

### Cell 4 · `lock_plan` (function)

In [4]:
def lock_plan(plan: Dict[str, Any]) -> Dict[str, Any]:
    """Validates structure and locks the plan against drift."""
    keys = {u["key"] for u in plan["units"]}
    if "objectives" not in keys:
        raise ValueError("Plan contract violation: mandatory unit 'objectives' missing")
    if not ({"methodology", "approach", "need-assessment"} & keys):
        raise ValueError("Plan contract violation: methodology/approach unit missing")
    if "references" not in keys:
        raise ValueError("Plan contract violation: mandatory unit 'references' missing")

    plan["locked"] = True
    serialized = json.dumps({"units": plan["units"], "acceptance": plan["acceptance"], "callId": plan["callId"]}, sort_keys=True)
    plan["checksum"] = hashlib.sha256(serialized.encode("utf-8")).hexdigest()
    return plan

### Cell 5 · `render_proposal_narrative` (function)

In [5]:
# ==============================================================================
# Skill 2: Format Skill (Render Proposal Narrative, Budget CSV, Compliance MD)
# ==============================================================================

def render_proposal_narrative(plan: Dict[str, Any], draft_sections: Dict[str, str]) -> str:
    """Formats full markdown proposal narrative with statutory AI-disclosure banner."""
    lines = [
        f"# {plan['title']}",
        f"**Target Agency Call:** {plan['callId']}",
        f"**Executing Institution:** {INSTITUTION['name']}",
        f"**Plan ID:** {plan['id']} (Checksum: {plan['checksum'][:12]}...)",
        "",
        "> **Notice:** AI-assisted drafting was utilized for initial synthesis. "
        "All scientific claims, methodology, and budgets are reviewed and verified by faculty.",
        "",
    ]
    for unit in plan["units"]:
        lines.append(f"## {unit['title']}")
        text = draft_sections.get(unit["key"], f"Section content for {unit['title']} meeting minimum {unit['min_words']} words.")
        lines.append(text)
        lines.append("")
    return "\n".join(lines)

### Cell 6 · `render_budget_csv` (function)

In [6]:
def render_budget_csv(budget_data: Dict[str, Any]) -> str:
    """Renders itemized CSV table of direct costs and overheads."""
    lines = ["Category,Item,Basis,Amount_INR"]
    for item in budget_data.get("items", []):
        lines.append(f"{item['category']},{item['item']},{item.get('basis', 'actual')},{item['amount']}")
    lines.append(f"Direct,Total Direct Costs,,{budget_data.get('direct', 0)}")
    lines.append(f"Overhead,Institutional Overhead (15%),,{budget_data.get('overhead', 0)}")
    lines.append(f"Total,Grand Total,,{budget_data.get('total', 0)}")
    return "\n".join(lines)

### Cell 7 · `render_compliance_markdown` (function)

In [7]:
def render_compliance_markdown(compliance_checks: List[Dict[str, Any]]) -> str:
    """Renders formatted compliance checklist table."""
    lines = [
        "# Compliance Review Checklist",
        "",
        "| Status | Check ID | Requirement | Evidence Found |",
        "| :--- | :--- | :--- | :--- |",
    ]
    for c in compliance_checks:
        lines.append(f"| {c['status'].upper()} | {c['id']} | {c['requirement']} | {c['evidence']} |")
    return "\n".join(lines)

### Cell 8 · `reusability_check` (function)

In [8]:
# ==============================================================================
# Reusability Verification (Concept 3)
# ==============================================================================

def reusability_check(plan_gov: Dict[str, Any], plan_ind: Dict[str, Any]):
    """
    Verifies that both plans conform to the exact same top-level schema
    despite serving completely different funding domains.
    """
    keys_gov = set(plan_gov.keys())
    keys_ind = set(plan_ind.keys())
    same_schema = keys_gov == keys_ind
    both_locked = plan_gov["locked"] and plan_ind["locked"]
    distinct_checksums = plan_gov["checksum"] != plan_ind["checksum"]

    print("\n=== Reusability Check Across Domains ===")
    print(f"Identical Schema: {same_schema} ({keys_gov})")
    print(f"Both Plans Cryptographically Locked: {both_locked}")
    print(f"Distinct Integrity Checksums: {distinct_checksums}")
    conclusion = "SKILL IS REUSABLE ACROSS DOMAINS" if same_schema and both_locked and distinct_checksums else "REUSABILITY FAILED"
    print(f"Conclusion: {conclusion}")
    assert conclusion == "SKILL IS REUSABLE ACROSS DOMAINS"

### Cell 9 · `run_assignment_1_multidomain` (function)

In [9]:
# ==============================================================================
# Graded Assignments
# ==============================================================================

def run_assignment_1_multidomain():
    """
    Assignment 1 -- Multi-Domain Plan Synthesis:
    Demonstrate generating an Industry/CSR plan with CSR-specific sections
    using the exact same skill function.
    """
    print("\n--- [Assignment 1] Industry/CSR Plan Synthesis ---")
    industry_inputs = {
        "researchArea": "Assistive Smart Mobility for Specially-Abled Students",
        "callId": "call-tata-csr",
        "domain": "industry",
        "sections": [
            "executive-summary", "need-assessment", "objectives",
            "methodology", "sustainability", "compliance", "references"
        ],
    }
    plan_ind = apply_plan_skill(industry_inputs)
    print(f"Generated Industry Plan: {plan_ind['id']}")
    print(f"Sections defined: {[u['key'] for u in plan_ind['units']]}")
    print(f"Plan Checksum: {plan_ind['checksum']}")
    return plan_ind

### Cell 10 · `run_assignment_2_contract_failure` (function)

In [10]:
def run_assignment_2_contract_failure():
    """
    Assignment 2 -- Test Against Missing Mandatory Unit:
    If a plan attempts to omit 'objectives', confirm the skill raises a
    contract violation error rather than outputting an invalid plan.
    """
    print("\n--- [Assignment 2] Contract Violation Test (Missing Objectives) ---")
    invalid_inputs = {
        "researchArea": "Autonomous Drones",
        "callId": "call-serb-crg",
        "domain": "government",
        "sections": ["abstract", "methodology", "references"],  # Missing 'objectives'!
    }
    try:
        apply_plan_skill(invalid_inputs)
        print("FAIL: Skill did not catch missing mandatory unit!")
    except ValueError as e:
        print(f"PASS: Contract violation caught as expected: '{e}'")

### Cell 11 · Run the lab

The lab demo and both graded assignments, with the console captured for the checks below.

In [11]:
# Console capture: everything the lab prints is shown below AND kept in memory,
# so the acceptance checks in the last cell can assert on the real run.
import io
import sys


class ConsoleTee(io.StringIO):
    """Mirror stdout into a buffer while the lab runs."""

    def __enter__(self):
        self._stdout = sys.stdout
        sys.stdout = self
        return self

    def __exit__(self, *exc):
        sys.stdout = self._stdout
        return False

    def write(self, text):
        self._stdout.write(text)
        return super().write(text)


console = ConsoleTee()

with console:
    print("======================================================================")
    print("Lab 3: Skill Creation -- Reusable Plan & Format Skills")
    print("======================================================================")

    # 1. Apply plan skill for Government Domain (SERB)
    gov_inputs = {
        "researchArea": "Federated Learning for Diabetic Retinopathy",
        "callId": "call-serb-crg",
        "domain": "government",
        "sections": [
            "abstract", "objectives", "background", "methodology",
            "timeline", "outcomes", "data-management", "budget-justification", "references"
        ],
    }
    plan_gov = apply_plan_skill(gov_inputs)
    print("Government Grant Plan:")
    print(f"ID: {plan_gov['id']}, Checksum: {plan_gov['checksum']}")
    print(f"Units Count: {len(plan_gov['units'])}, Locked: {plan_gov['locked']}")

    # 2. Assignment 1: Generate Industry Plan
    plan_ind = run_assignment_1_multidomain()

    # 3. Check reusability
    reusability_check(plan_gov, plan_ind)

    # 4. Demonstrate Format Skill
    print("\n=== Formatted Proposal Narrative Sample ===")
    sample_text = render_proposal_narrative(
        plan_gov,
        {
            "abstract": "This project establishes a four-clinic federated screening pipeline for retinopathy...",
            "objectives": "1. Deploy edge fundus grading. 2. Evaluate clinical sensitivity in 2,400 rural patients.",
        }
    )
    print(sample_text[:400] + "...\n[Remaining sections rendered]")

    # 5. Assignment 2: Contract violation test
    run_assignment_2_contract_failure()

Lab 3: Skill Creation -- Reusable Plan & Format Skills
Government Grant Plan:
ID: PLAN-GOVERNMENT-1625, Checksum: e3d9ff97d59a06610b90ec93c28d60ed6cd176c30c9dd6d0dc8e76a7e0717d1a
Units Count: 9, Locked: True

--- [Assignment 1] Industry/CSR Plan Synthesis ---
Generated Industry Plan: PLAN-INDUSTRY-5241
Sections defined: ['executive-summary', 'need-assessment', 'objectives', 'methodology', 'sustainability', 'compliance', 'references']
Plan Checksum: e396362fd2bb12ab1e719f399e89a88e2e3b8b1e23a5f9cb29e185bf2330a771

=== Reusability Check Across Domains ===
Identical Schema: True ({'units', 'domain', 'locked', 'id', 'version', 'title', 'callId', 'acceptance', 'phases', 'checksum'})
Both Plans Cryptographically Locked: True
Distinct Integrity Checksums: True
Conclusion: SKILL IS REUSABLE ACROSS DOMAINS

=== Formatted Proposal Narrative Sample ===
# Grant Proposal Plan -- Federated Learning for Diabetic Retinopathy
**Target Agency Call:** call-serb-crg
**Executing Institution:** Meridian Ins

### Cell 12 · Acceptance checks and deliverables

Each check asserts on the trace of the run above. The trace and the check report are exported to `lab3_artifacts/` (committed under `Course-Project/evidence/`).

In [12]:
import json
import re
from pathlib import Path

LAB = 3
CHECKS = [('Government plan is produced and locked', 'Locked: True', 1),
    ('Same plan skill serves the Industry/CSR domain', 'Generated Industry Plan: PLAN-INDUSTRY', 1),
    ('Both domains share one output schema', 'Identical Schema: True', 1),
    ('Each locked plan has its own integrity checksum', 'Distinct Integrity Checksums: True', 1),
    ('Skill is declared reusable across domains', 'SKILL IS REUSABLE ACROSS DOMAINS', 1),
    ('Format skill discloses AI-assisted drafting', 'AI-assisted drafting', 1),
    ('Contract violation (missing objectives) is refused', 'PASS: Contract violation caught as expected', 1)]

trace = console.getvalue()
universal = [
    ("Run produced console output", r"\S", 1),
    ("Graded Assignment 1 executed", r"\[Assignment 1\]", 1),
    ("Graded Assignment 2 executed", r"\[Assignment 2\]", 1),
]
results = []
for label, pattern, minimum in universal + CHECKS:
    found = len(re.findall(pattern, trace, flags=re.MULTILINE))
    results.append({"check": label, "pattern": pattern, "expected_min": minimum,
                    "found": found, "passed": found >= minimum})
no_errors = "Traceback (most recent call last)" not in trace
results.append({"check": "No unhandled exception in the run", "pattern": "Traceback",
                "expected_min": 0, "found": 0 if no_errors else 1, "passed": no_errors})

for i, r in enumerate(results, 1):
    print(f"[{'PASS' if r['passed'] else 'FAIL'}] {i:>2}. {r['check']}")
passed = sum(r["passed"] for r in results)
print(f"\nLab {LAB}: {passed}/{len(results)} acceptance checks passed")

outdir = Path(f"lab{LAB}_artifacts")
outdir.mkdir(exist_ok=True)
(outdir / f"lab{LAB}_console_trace.txt").write_text(trace, encoding="utf-8")
(outdir / f"lab{LAB}_acceptance_checks.json").write_text(
    json.dumps({"lab": LAB, "passed": passed, "total": len(results), "checks": results}, indent=2),
    encoding="utf-8")
print("Artifacts written to", outdir.as_posix())

assert passed == len(results), "acceptance checks failed"

[PASS]  1. Run produced console output
[PASS]  2. Graded Assignment 1 executed
[PASS]  3. Graded Assignment 2 executed
[PASS]  4. Government plan is produced and locked
[PASS]  5. Same plan skill serves the Industry/CSR domain
[PASS]  6. Both domains share one output schema
[PASS]  7. Each locked plan has its own integrity checksum
[PASS]  8. Skill is declared reusable across domains
[PASS]  9. Format skill discloses AI-assisted drafting
[PASS] 10. Contract violation (missing objectives) is refused
[PASS] 11. No unhandled exception in the run

Lab 3: 11/11 acceptance checks passed
Artifacts written to lab3_artifacts
